# 10 — Baseline full pipeline

Единый baseline notebook (объединение шагов 01..04) с сохранением результатов в `baseline_v1` и печатью ключевых численных метрик.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.basis import get_basis
from src.control import candidate_gains, closed_loop_jacobian
from src.experiments import compute_tail_metrics, summarize_tail_metrics, is_hurwitz, residual_statistics, add_gaussian_noise
from src.identification import fit_identified_model, predict_vector_field, rmse
from src.lyapunov import numerical_jacobian, solve_lyapunov, lyapunov_value, evaluate_lyapunov_grid
from src.simulation import simulate_batch, uncertain_dynamics
from src.systems import cross_coupled_uncontrolled
from src.uncertainty import compute_residuals, estimate_epsilon, bounded_disturbance, compute_bounding_box
from src.utils import ensure_dir, save_dataframe, save_json, get_repo_root, set_seed

ROOT = get_repo_root()


In [ ]:
set_seed(42)
BASE_RESULTS = ROOT / 'results' / 'baseline_v1'
BASE_DATA = ROOT / 'data' / 'processed' / 'baseline_v1'
for p in [BASE_RESULTS / 'figures', BASE_RESULTS / 'tables', BASE_RESULTS / 'metrics', BASE_DATA]:
    ensure_dir(p)

initials = np.array([[-1.2, -0.8], [-1.0, 0.7], [-0.6, 1.1], [0.5, -1.0], [1.0, 0.9], [1.3, -0.4]], dtype=float)
t_eval = np.linspace(0.0, 10.0, 400)
traj_true = simulate_batch(cross_coupled_uncontrolled, initials, (0.0, 10.0), t_eval)
X = np.vstack([states for _, states in traj_true])
Xdot = np.vstack([np.array([cross_coupled_uncontrolled(float(t), s) for t, s in zip(ts, states)]) for ts, states in traj_true])
save_dataframe(pd.DataFrame(np.hstack([X, Xdot]), columns=['x1', 'x2', 'xdot1', 'xdot2']), BASE_DATA / 'cross_coupled_dataset.csv')

plt.figure()
for _, st in traj_true:
    plt.plot(st[:,0], st[:,1], lw=1.2)
plt.xlabel('x1'); plt.ylabel('x2'); plt.title('Baseline true phase trajectories')
plt.tight_layout(); plt.savefig(BASE_RESULTS / 'figures' / 'phase_true.png', dpi=220); plt.show()


In [ ]:
basis_name = 'quadratic_full'
basis_fn = get_basis(basis_name)
fit = fit_identified_model(X, Xdot, basis_fn, basis_name=basis_name)
Xdot_hat = predict_vector_field(X, basis_fn, fit.coefficients)
res = compute_residuals(Xdot, Xdot_hat)
res_norms = np.linalg.norm(res, axis=1)
epsilon = estimate_epsilon(res_norms, q=0.95)
omega_lower, omega_upper = compute_bounding_box(X)

fhat = lambda t, x: predict_vector_field(x[None, :], basis_fn, fit.coefficients)[0]
A = numerical_jacobian(lambda x: fhat(0.0, x), np.zeros(2))
eig_A = np.linalg.eigvals(A)
P = solve_lyapunov(A, np.eye(2))
eig_P = np.linalg.eigvalsh(P)

print('RMSE:', rmse(Xdot, Xdot_hat))
print('MAE:', float(np.mean(np.abs(Xdot - Xdot_hat))))
print('epsilon (q95):', epsilon)
print('Omega lower:', omega_lower)
print('Omega upper:', omega_upper)
print('A =
', A)
print('eig(A) =', eig_A)
print('P =
', P)
print('P positive definite:', np.all(eig_P > 0))
print('V([1,1]) =', lyapunov_value(np.array([1.0, 1.0]), P))

plt.figure(); plt.hist(res_norms, bins=30, alpha=0.8, color='steelblue'); plt.axvline(epsilon, color='crimson', ls='--', label=f'epsilon={epsilon:.3f}'); plt.legend(); plt.xlabel('||r||'); plt.ylabel('Count'); plt.title('Baseline residual histogram'); plt.tight_layout(); plt.savefig(BASE_RESULTS / 'figures' / 'residual_hist.png', dpi=220); plt.show()

traj_hat = simulate_batch(fhat, initials, (0,10), t_eval)
plt.figure()
for _, st in traj_hat:
    plt.plot(st[:,0], st[:,1], lw=1.2)
plt.xlabel('x1'); plt.ylabel('x2'); plt.title('Baseline identified phase trajectories')
plt.tight_layout(); plt.savefig(BASE_RESULTS / 'figures' / 'phase_identified.png', dpi=220); plt.show()

xx, yy, vv = evaluate_lyapunov_grid(P, (-1.5, 1.5), (-1.5, 1.5), points=90)
plt.figure(); cs = plt.contour(xx, yy, vv, levels=14, cmap='viridis'); plt.clabel(cs, inline=True, fontsize=7)
plt.xlabel('x1'); plt.ylabel('x2'); plt.title('Baseline Lyapunov level sets')
plt.tight_layout(); plt.savefig(BASE_RESULTS / 'figures' / 'lyapunov_level_sets.png', dpi=220); plt.show()


In [ ]:
B = np.array([[0.0], [1.0]])
initials_unc = np.array([[-1.0,-0.8],[-0.8,1.0],[0.8,-1.0],[1.1,0.9]], dtype=float)
t_unc = np.linspace(0, 12, 500)

unc_dyn = uncertain_dynamics(fhat, lambda t: bounded_disturbance(t, epsilon))
traj_unc = simulate_batch(unc_dyn, initials_unc, (0,12), t_unc)
unc_tail = [compute_tail_metrics(st) for _, st in traj_unc]
unc_summary = summarize_tail_metrics(unc_tail)
print('Uncontrolled summary:', unc_summary)

plt.figure()
for _, st in traj_unc:
    plt.plot(st[:,0], st[:,1], lw=1.2)
plt.xlabel('x1'); plt.ylabel('x2'); plt.title('Baseline uncertain uncontrolled')
plt.tight_layout(); plt.savefig(BASE_RESULTS / 'figures' / 'uncertain_uncontrolled.png', dpi=220); plt.show()

closed_loop = {}
plt.figure()
for gain_name, K in candidate_gains().items():
    A_cl = closed_loop_jacobian(A, B, K)
    eig_A_cl = np.linalg.eigvals(A_cl)
    cl_dyn = uncertain_dynamics(lambda t, x, K=K: fhat(t, x) + (B @ (K @ x)).reshape(-1), lambda t: bounded_disturbance(t, epsilon))
    traj_cl = simulate_batch(cl_dyn, initials_unc, (0,12), t_unc)
    cl_tail = summarize_tail_metrics([compute_tail_metrics(st) for _, st in traj_cl])
    closed_loop[gain_name] = {
        'K': K.tolist(),
        'A_cl': A_cl.tolist(),
        'eig_A_cl_real_parts': np.real(eig_A_cl).tolist(),
        'stable_closed_loop': is_hurwitz(A_cl),
        **cl_tail,
    }
    print(f'Gain {gain_name}: eig(A_cl)={eig_A_cl}, summary={cl_tail}')
    for i, (_, st) in enumerate(traj_cl):
        plt.plot(st[:,0], st[:,1], lw=1.0, label=gain_name if i == 0 else None)

plt.xlabel('x1'); plt.ylabel('x2'); plt.title('Baseline controlled uncertain comparison'); plt.legend()
plt.tight_layout(); plt.savefig(BASE_RESULTS / 'figures' / 'uncertain_controlled_comparison.png', dpi=220); plt.show()

summary = {
    'dictionary_name': basis_name,
    'rmse': rmse(Xdot, Xdot_hat),
    'mae': float(np.mean(np.abs(Xdot - Xdot_hat))),
    'epsilon_q95': float(epsilon),
    'omega_lower': omega_lower.tolist(),
    'omega_upper': omega_upper.tolist(),
    'A': A.tolist(),
    'eig_A_real_parts': np.real(eig_A).tolist(),
    'open_loop_stable': is_hurwitz(A),
    'P': P.tolist(),
    'P_positive_definite': bool(np.all(eig_P > 0)),
    'V_at_[1,1]': float(lyapunov_value(np.array([1.0, 1.0]), P)),
    'residual_stats': residual_statistics(res),
    'uncontrolled': unc_summary,
    'K_candidates': {name: arr.tolist() for name, arr in candidate_gains().items()},
    'closed_loop': closed_loop,
}
save_json(summary, BASE_RESULTS / 'metrics' / 'baseline_summary.json')

summary_row = {
    'dictionary_name': summary['dictionary_name'],
    'rmse': summary['rmse'],
    'mae': summary['mae'],
    'epsilon_q95': summary['epsilon_q95'],
    'open_loop_stable': summary['open_loop_stable'],
    'open_loop_ultimate_radius': summary['uncontrolled']['ultimate_radius_estimate'],
    'mild_ultimate_radius': summary['closed_loop']['mild']['ultimate_radius_estimate'],
    'medium_ultimate_radius': summary['closed_loop']['medium']['ultimate_radius_estimate'],
    'aggressive_ultimate_radius': summary['closed_loop']['aggressive']['ultimate_radius_estimate'],
}
pd.DataFrame([summary_row]).to_csv(BASE_RESULTS / 'tables' / 'baseline_summary.csv', index=False)

print('
Saved baseline artifacts to:', BASE_RESULTS)
print('Saved baseline dataset to:', BASE_DATA)
pd.DataFrame([summary_row])
